# Introduction

## 1. What is Collaborative Filtering?
Collaborative Filtering (CF) is a fundamental technique in recommender systems that automates the process of predicting a user's preference or rating for a specific item based on the observed preferences of a community of other users. Unlike content-based filtering, which relies on the intrinsic properties of items (e.g., genre, color, keywords), collaborative filtering relies solely on past user-item interactions. The foundational assumption is that if User A and User B have historically agreed on the quality of several items, they are likely to agree on a new, unseen item in the future.

## 2. Is it about User-to-User, Item-to-Item, or User-to-Item Similarity?
Collaborative Filtering encompasses **all three perspectives**, depending on the specific algorithmic approach adopted:
- **User-to-User Similarity (Memory-Based CF):** This approach directly compares the rating vectors of users. If the system needs to recommend an item to User $u$, it finds $k$ other users whose rating history is most similar to $u$'s (neighbors).
- **Item-to-Item Similarity (Memory-Based CF):** This approach transposes the matrix and compares items based on how users have rated them. If a user liked Item $i$, the system recommends other items that have the most similar rating profile to $i$. This was famously popularized by Amazon's "Customers Who Bought This Item Also Bought" feature.
- **User-to-Item Relationship (Model-Based CF / Matrix Factorization):** This approach does not compute explicit similarity between raw rows or columns. Instead, it models the interaction as an inner product of latent factors—a **User-to-Item** affinity. It discovers abstract features that explain the observed ratings, such that the predicted rating $\hat{r}_{ui}$ is a function of the user's preference vector and the item's characteristic vector.

## 3. When Exactly is the Proper Case and Time to Use Collaborative Filtering?
Collaborative Filtering is the appropriate methodology under the following specific conditions:
- **Sparse User Feedback:** You have a large user base and a large item catalog, but each individual user has only interacted with a tiny fraction of the total items.
- **Subjective or Aesthetic Domains:** The items lack objective, machine-readable attributes (e.g., movies, music, jokes, handmade crafts). For example, you cannot analyze the "pixel data" of a movie to determine if it is "funny"; you must rely on what other people who laughed at the same things thought.
- **Cross-Category Discovery:** You want to surprise the user with recommendations from categories they have never visited (serendipity). For instance, a user who likes *Blade Runner* might like a specific Philip K. Dick novel, an association only visible through overlapping fanbases, not through text analysis.

## 4. Where Can We Implement Collaborative Filtering?
Collaborative Filtering is implemented across a wide spectrum of digital services where user decision fatigue is high:
- **E-commerce Platforms:** Amazon, eBay, and Etsy use it for product recommendations on detail pages and checkout cross-sells.
- **Streaming Media Services:** Netflix (movie suggestions), Spotify (Discover Weekly playlist), and YouTube (video recommendations) rely heavily on CF to keep users engaged by surfacing deep catalog content.
- **Social Networks:** LinkedIn ("People You May Know"), Facebook (Friend Suggestions), and Twitter (Who to Follow) treat "following a user" as the item interaction.
- **Digital Advertising:** Ad exchanges use CF to predict Click-Through Rate (CTR) for a specific user-ad pair based on similar user cohorts.

## 5. Why Should We Use Collaborative Filtering?
The primary justification for using Collaborative Filtering is **domain independence and discovery**. We should use it because:
1.  **No Feature Engineering Required:** It eliminates the need for subject matter experts to manually tag every song with "distorted guitar" or every book with "unreliable narrator." The system learns these latent descriptors from the data itself.
2.  **Quality of Insight:** It captures nuanced, hard-to-quantify preferences (e.g., "Movies with a bittersweet ending").
3.  **Scalability of Model Training:** Modern matrix factorization algorithms are highly parallelizable and can handle matrices with hundreds of millions of rows and columns efficiently using Stochastic Gradient Descent (SGD) or Alternating Least Squares (ALS).

## 6. Step-by-Step Mathematical Workflow with Formulation
When we have the data, specifically a set of triplets $(u, i, r_{ui})$ where $u$ is user ID, $i$ is item ID, and $r_{ui}$ is the interaction strength (explicit rating or implicit count). We work with **Model-Based Collaborative Filtering (Matrix Factorization)** as follows:

### **Step 1: Define the Interaction Matrix $R$**
We construct a sparse matrix $R \in \mathbb{R}^{m \times n}$, where $m$ is the number of users and $n$ is the number of items. Most entries $r_{ui}$ are missing (NaN).

### **Step 2: Mathematical Model Definition (Hypothesis)**
We assume each user $u$ can be represented by a latent vector $p_u \in \mathbb{R}^k$ and each item $i$ by a latent vector $q_i \in \mathbb{R}^k$. Here, $k$ is the number of latent factors (dimensionality of the abstract space, e.g., $k=50$ or $100$).
The predicted rating $\hat{r}_{ui}$ is modeled as the dot product (interaction) between the user and item vectors:
$$
\hat{r}_{ui} = q_i^T p_u = \sum_{f=1}^{k} q_{if} \cdot p_{uf}
$$

### **Step 3: Define the Loss Function (Objective)**
We need to minimize the difference between the observed ratings $r_{ui}$ and the predicted ratings $\hat{r}_{ui}$. To prevent overfitting to the sparse data, we introduce **L2 Regularization**. The objective function $\mathcal{L}$ to minimize is:
$$
\mathcal{L} = \min_{p^*, q^*} \sum_{(u,i) \in \mathcal{K}} \left( r_{ui} - q_i^T p_u \right)^2 + \lambda \left( \|q_i\|^2 + \|p_u\|^2 \right)
$$

Where:
- $\mathcal{K}$ is the set of known (user, item) pairs in the training data.
- $\lambda$ is the regularization hyperparameter controlling the penalty on vector magnitude.

### **Step 4: Optimization via Stochastic Gradient Descent (SGD)**
We iterate over each known rating $r_{ui}$ in the training set. We compute the prediction error:
$$
e_{ui} = r_{ui} - q_i^T p_u
$$

We then compute the gradients of the loss with respect to the parameters and update them in the opposite direction of the gradient. **Update for User Vector $p_u$:**
$$
p_u \leftarrow p_u + \eta \cdot (e_{ui} \cdot q_i - \lambda \cdot p_u)
$$

**Update for Item Vector $q_i$:**
$$
q_i \leftarrow q_i + \eta \cdot (e_{ui} \cdot p_u - \lambda \cdot q_i)
$$

Where $\eta$ is the learning rate.

### **Step 5: Prediction for Missing Values**
After convergence (or a fixed number of epochs), we reconstruct the full matrix $\hat{R}$. For a user $u$ and an unseen item $j$, the final predicted score is:
$$
\hat{r}_{uj} = q_j^T p_u
$$
We then recommend the top-$N$ items with the highest $\hat{r}_{uj}$ values that the user has not yet interacted with.

# Module Loading

In [3]:
import gdown
import os
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pylab import rcParams
import pandas as pd
import numpy as np
from google.colab import drive
from warnings import filterwarnings
import duckdb
import pandas as pd
from typing import Optional, Union, List, Dict, Any, ContextManager
from pathlib import Path
from contextlib import contextmanager
filterwarnings('ignore')

In [5]:
# Modern Professional Color Palette
modern_colors = [
    "#1f77b4",   # Vibrant Blue (Primary)
    "#ff7f0e",   # Bright Orange (Accent/Comparison)
    "#2ca02c",   # Fresh Green (Success/Positive)
    "#d62728",   # Soft Red (Alert/Warning)
    "#9467bd",   # Elegant Purple
    "#8c564b",   # Warm Brown
    "#e377c2",   # Pink
    "#7f7f7f",   # Neutral Gray
    "#bcbd22",   # Olive/Yellow-Green
    "#17becf"    # Cyan/Teal
]

# 2. Main Style Dictionary
modern_light_style = {
    # Background - Clean and bright
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#f8f9fa",
    "savefig.facecolor": "#ffffff",

    # Grid - Very subtle and non-distracting
    "axes.grid": True,
    "grid.color": "#e6e8eb",
    "grid.linestyle": "--",
    "grid.linewidth": 0.8,
    "axes.grid.which": "both",

    # Typography
    "text.color": "#1f2937",
    "axes.labelcolor": "#1f2937",
    "xtick.color": "#374151",
    "ytick.color": "#374151",
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.titlepad": 18,
    "font.size": 12,
    "font.family": "sans-serif", # You can change to 'Arial', 'Helvetica', etc.

    # Spines - Clean and minimal
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.edgecolor": "#4b5563",
    "axes.linewidth": 1.2,

    # Lines and markers
    "axes.prop_cycle": plt.cycler(color=modern_colors),
    "lines.linewidth": 2.5,
    "lines.markersize": 7,
    "lines.markeredgewidth": 0.8,

    # Patches (bars, areas, etc.)
    "patch.edgecolor": "#ffffff",
    "patch.linewidth": 0.8,

    # Legend
    "legend.frameon": False,
    "legend.loc": "best",
    "legend.fontsize": 11,
}

plt.style.use('default')
sns.set_theme(style="whitegrid", rc=modern_light_style)
plt.rcParams.update(modern_light_style)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=modern_colors)

In [3]:
class DuckDBManager:
    """
    Manager class for DuckDB connections and queries.

    Example:
        db = DuckDBManager('mydb.duckdb')
        df = db.query("SELECT * FROM users")
        db.close()

        # Or use context manager:
        with DuckDBManager('mydb.duckdb') as db:
            df = db.query("SELECT * FROM users")
    """

    def __init__(self,
                 db_path: Union[str, Path],
                 read_only: bool = True,
                 threads: Optional[int] = None,
                 memory_limit: Optional[str] = '2GB'):
        """
        Initialize DuckDB manager.

        Args:
            db_path: Path to database file (use ':memory:' for in-memory DB)
            read_only: Open in read-only mode
            threads: Number of CPU threads to use
            memory_limit: Memory limit (e.g., '4GB', '1TB')
        """
        self.db_path = str(db_path)
        self.read_only = read_only
        self.conn = None

        # Create connection
        self._create_connection(threads, memory_limit)

    def _create_connection(self, threads: Optional[int] = None,
                          memory_limit: Optional[str] = None):
        """Create DuckDB connection with optional settings."""
        self.conn = duckdb.connect(database=self.db_path, read_only=self.read_only)

        # Configure settings
        if threads:
            self.conn.execute(f"SET threads = {threads}")
        if memory_limit:
            self.conn.execute(f"SET memory_limit = '{memory_limit}'")

    def query(self,
              query: str,
              params: Optional[Union[List, Dict, tuple]] = None,
              fetch_size: Optional[int] = None) -> pd.DataFrame:
        """
        Execute query and return DataFrame.

        Args:
            query: SQL query string
            params: Query parameters
            fetch_size: Number of rows to fetch (None for all)

        Returns:
            pandas DataFrame
        """
        if self.conn is None:
            raise ValueError("Connection is closed. Please reconnect.")

        try:
            if params is not None:
                result = self.conn.execute(query, params)
            else:
                result = self.conn.execute(query)

            if fetch_size:
                return result.fetch_df_chunk(fetch_size)
            return result.fetchdf()

        except Exception as e:
            raise Exception(f"Query failed: {e}\nQuery: {query}")

    def query_arrow(self, query: str, params: Optional[Union[List, Dict]] = None):
        """Execute query and return Arrow table (faster for large datasets)."""
        if params is not None:
            return self.conn.execute(query, params).fetch_arrow_table()
        return self.conn.execute(query).fetch_arrow_table()

    def register_dataframe(self, name: str, df: pd.DataFrame):
        """Register pandas DataFrame as temporary table (table_view)."""
        self.conn.register(name, df)

    def ListedTable(self):
        tables = self.conn.execute("SELECT table_name FROM duckdb_tables()").df()
        data = list()
        for table in tables['table_name']:
            count = self.conn.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
            data.append({'table_name': table, 'row_count': count})
        info_df = pd.DataFrame(data)
        info_df = info_df.sort_values(by='row_count', ascending=False)
        info_df['row_count'] = info_df['row_count'].apply(lambda x: f"{x:,}")
        display(info_df)
        return info_df

    def execute(self, query: str, params: Optional[Union[List, Dict]] = None):
        """Execute query without returning results."""
        if params is not None:
            self.conn.execute(query, params)
        else:
            self.conn.execute(query)

    def table_exists(self, table_name: str) -> bool:
        """Check if table exists in database."""
        result = self.query(
            "SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?",
            [table_name]
        )
        return result.iloc[0, 0] > 0

    def get_tables(self) -> List[str]:
        """Get list of all tables in database."""
        df = self.query("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'")
        return df['table_name'].tolist()

    def get_schema(self, table_name: str) -> pd.DataFrame:
        """Get schema information for a table."""
        return self.query(f"DESCRIBE {table_name}")

    def close(self):
        """Close database connection."""
        if self.conn:
            self.conn.close()
            self.conn = None

    def __enter__(self):
        """Context manager entry."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit."""
        self.close()


In [4]:
@contextmanager
def duckdb_connection(db_path: Union[str, Path],
                      read_only: bool = False,
                      **kwargs) -> DuckDBManager:
    """
    Context manager for DuckDB connection.

    Example:
        with duckdb_connection('mydb.duckdb') as db:
            df = db.query("SELECT * FROM users")
    """
    db = DuckDBManager(db_path, read_only, **kwargs)
    try:
        yield db
    finally:
        db.close()

# Data Loading

## Download Data

In [16]:
def glink(uid : str, 
          thepath : str, 
          verbose : bool = False,
         ) -> str:
    coks = re.search(r'[-\w]{25,}', uid)
    file_id = coks.group(0) if coks else uid
    gdown.download(id=file_id, output=str(thepath), quiet = not verbose)
    thepath = os.path.realpath(thepath)
    
    # Verify the download isn't a tiny HTML
    if os.path.exists(thepath):
        file_size_kb = os.path.getsize(thepath) / 1024
        if file_size_kb < 100:
            print(f"Warning: {thepath} is very small ({file_size_kb:.2f} KB).")
            print("Check if the Google Drive link is set to 'Anyone with the link'.")
    return thepath

In [17]:
# Data GA4
url_ga4 = '1GRTBYqA0iKLLFflzcfNo1ctVWzis872Q'
dbfilename = './DB_ga4_ecommerce.duckdb'
ga4dbpath = glink(url_ga4, dbfilename)

In [18]:
#Data FPgrowth Prediction
url_fpg = '1YZN36UfzIWZ_Fy6ManAnQxeYjSBHA8Fn'
fpgname = './20260418_fpgrowth_prediction.zip'
fpgdbpath = glink(url_fpg, fpgname)

In [19]:
print(f'GApath : {ga4dbpath}.\n')
print(f'FP Growth Prediction : {fpgdbpath}.')

GApath : /kaggle/working/DB_ga4_ecommerce.duckdb.

FP Growth Prediction : /kaggle/working/20260418_fpgrowth_prediction.zip.


In [20]:
db = DuckDBManager(db_path = ga4dbpath)
_ = db.ListedTable()

mas_train = db.query('SELECT * FROM FinalMasterData')

,table_name,row_count
0,FinalMasterData,"1,014,426"
13,MasterTrainData,"1,014,426"
2,FinalMasterData_train,"811,540"
1,FinalMasterData_test,"202,886"
14,UserFeature_Analysis,"47,474"
3,fpgrowth_transactions,"46,375"
12,mapping_product_name,429
9,mapping_primary_country,109
7,mapping_department_id,81
5,mapping_brand,7


In [22]:
predpath = os.path.dirname(ga4dbpath)
if 'predpath' in locals() or 'predpath' in globals():
    print(f"Searching for ZIP files in: {predpath}")
    zip_files = [f for f in os.listdir(predpath) if f.endswith('.zip')]

    if zip_files:
        print("Found zip files:")
        for jf in zip_files:
            print(os.path.join(predpath, jf))
    else:
        print("No zip files found in this directory.")
else:
    print("Error: 'predpath' variable is not defined. Please ensure the previous cell setting predpath was executed.")

Searching for ZIP files in: /kaggle/working
Found zip files:
/kaggle/working/20260418_fpgrowth_prediction.zip


In [24]:
from shutil import copy
from zipfile import ZipFile

destination_dir = '/kaggle/working/fpgrowth_data'
os.makedirs(destination_dir, exist_ok = True)
all_extracted_json_data = list()

print(f"Processing ZIP files from {predpath}.")
for zip_file_name in zip_files:
    src_zip_path = os.path.join(predpath, zip_file_name)
    dest_zip_path = os.path.join(destination_dir, zip_file_name)

    if not os.path.exists(src_zip_path):
        print(f"Warning: Source ZIP file not found: {src_zip_path}. Skipping.")
        continue
    copy(src_zip_path, dest_zip_path)
    with ZipFile(dest_zip_path, 'r') as zip_ref:
        zip_ref.extractall(destination_dir)

Processing ZIP files from /kaggle/working.


In [30]:
import json
jsonfile = [os.path.join(destination_dir, f) for f in os.listdir(destination_dir) if f.endswith('.json')]
choosejson = jsonfile[-1]

with open(choosejson, 'r') as f:
    fpgrowthdata = json.load(f)

In [31]:
from itertools import islice

def dictslice(jsonfile:dict,
              count : int = 10,
              show : bool = True) -> dict:
    Data = dict(islice(jsonfile.items(), int(count)))
    if show:
        print(Data)
    return Data

a = dictslice(fpgrowthdata, 3)

{'1000684.125': [{'items': ['9196908'], 'support': 14}, {'items': ['9199084'], 'support': 44}], '1001326.125': [{'items': ['9197398'], 'support': 34}, {'items': ['9197946'], 'support': 10}], '1010983.25': [{'items': ['9196832'], 'support': 25}]}


In [32]:
def Json2Dataframe(jsondata : dict) -> pd.DataFrame:
    records = ({"group_id": group_id,
                "items"   : entry["items"],
                "support" : entry["support"]}
        for group_id, entries in jsondata.items()
        for entry in entries)
    data = pd.DataFrame.from_records(records)
    data["items"] = data["items"].apply(
        lambda x: ", ".join(x) if isinstance(x, (list, set)) else x)
    return data

DataFPGpredict = Json2Dataframe(fpgrowthdata)
display(DataFPGpredict.describe(include='object'))
print('\n\n')
display(DataFPGpredict.describe())

,group_id,items
count,27290,27290
unique,10631,677
top,59061040.0,9188192
freq,10,1958


,support
count,27290.000000
mean,48.772774
std,37.048558
min,8.000000
25%,19.000000
50%,38.000000
75%,72.000000
max,178.000000


In [33]:
print(mas_train.info())
#print(mas_train.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014426 entries, 0 to 1014425
Data columns (total 42 columns):
 #   Column                             Non-Null Count    Dtype  
---  ------                             --------------    -----  
 0   user_id                            1014426 non-null  float32
 1   product_id                         1014426 non-null  object 
 2   department_id                      1014426 non-null  int32  
 3   aisle_id                           1014426 non-null  int32  
 4   brand                              1014426 non-null  int32  
 5   category3                          1014426 non-null  int32  
 6   avg_interaction_price              1014426 non-null  float32
 7   view_count                         1014426 non-null  int32  
 8   add_to_cart_count                  1014426 non-null  int32  
 9   begin_checkout_count               1014426 non-null  int32  
 10  purchase_count                     1014426 non-null  int32  
 11  total_quantity          

## Notes

For variable data, we can refer to `DataFPGpredict` and `mas_train`!!!

# **Collaborative Filtering**

## Cython Optimized

In [6]:
os.makedirs('./cy', exist_ok = True)

In [7]:
%%writefile ./cy/_cy_evaluate.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_evaluate.pyx
# ROC-AUC computation from pre-computed item ranks for arycolbring.
# Parallelised per-user with OpenMP prange.

from libc.stdio cimport fprintf, stderr
from cython.parallel cimport prange, parallel

from _cy_types cimport CSRMatrix, flt
from _cy_math  cimport flt_compare, qsort


def calculate_auc_from_rank(CSRMatrix ranks,
                             int[::1]  num_train_positives,
                             flt[::1]  rank_data,
                             flt[::1]  auc,
                             int       num_threads):
    """
    Convert per-user item ranks (from predict_ranks) to ROC-AUC scores.

    For each user the AUC is the probability that a randomly chosen
    positive example is ranked above a randomly chosen negative.
    A perfect model scores 1.0; random scoring gives 0.5.

    Parameters
    ----------
    ranks               : CSRMatrix  — sparse rank matrix (users × items)
    num_train_positives : int32 array [n_users]  — # train positives per user
                          (excluded from the negative count)
    rank_data           : float32 array  — the .data buffer of `ranks`
                          (modified in-place: sorted per-user row)
    auc                 : float32 array [n_users]  — output AUC per user
    num_threads         : int ≥ 1
    """
    fprintf(stderr,
            b"[DEBUG] calculate_auc_from_rank: n_users=%d num_threads=%d\n",
            ranks.rows, num_threads)

    cdef int i, user_id, row_start, row_stop
    cdef int num_negatives, num_positives
    cdef flt rank

    with nogil, parallel(num_threads=num_threads):
        for user_id in prange(ranks.rows, schedule='static'):
            row_start     = ranks.get_row_start(user_id)
            row_stop      = ranks.get_row_end(user_id)
            num_positives = row_stop - row_start
            num_negatives = (ranks.cols
                             - num_positives
                             - num_train_positives[user_id])

            # Degenerate case: only one class present → return 0.5
            if num_positives == 0 or num_negatives == ranks.cols:
                auc[user_id] = 0.5
                continue

            # Sort positive ranks ascending so we can correct for ties
            qsort(&rank_data[row_start],
                  num_positives,
                  sizeof(flt),
                  flt_compare)

            for i in range(num_positives):
                rank = ranks.data[row_start + i]

                # Subtract the i other positives that rank above this one.
                # Clamp to zero to avoid negative ranks.
                rank = rank - i
                if rank < 0:
                    rank = 0

                # P(positive ranked above random negative) for this item
                auc[user_id] += 1.0 - rank / num_negatives

            if num_positives != 0:
                auc[user_id] /= num_positives

    fprintf(stderr, b"[DEBUG] calculate_auc_from_rank: done\n")

Writing ./cy/_cy_evaluate.pyx


In [9]:
%%writefile ./cy/_cy_fit_bpr.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_bpr.pyx
# BPR (Bayesian Personalised Ranking) training kernel for arycolbring.

import numpy as np
from libc.stdlib  cimport malloc, free
from libc.stdio   cimport fprintf, stderr
from cython.parallel cimport prange, parallel, threadid

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport rand_r, in_positives, sigmoid
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport warp_update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, \
                                 omp_init_lock, omp_destroy_lock

cdef double MAX_REG_SCALE = 1000000.0


def fit_bpr(CSRMatrix item_features,
            CSRMatrix user_features,
            CSRMatrix interactions,
            int[::1] user_ids,
            int[::1] item_ids,
            flt[::1] Y,
            flt[::1] sample_weight,
            int[::1] shuffle_indices,
            FastAryColBring model,
            double learning_rate,
            double item_alpha,
            double user_alpha,
            int num_threads,
            random_state):
    """
    One epoch of BPR-loss collaborative filtering.

    For each positive, one hard negative is sampled and a pairwise
    gradient update (same kernel as WARP) is applied with a sigmoid loss.
    """
    fprintf(stderr,
            b"[DEBUG] fit_bpr: no_examples=%d num_threads=%d\n",
            Y.shape[0], num_threads)

    cdef int i, j, no_examples, user_id, positive_item_id, negative_item_id
    cdef int sampled, row
    cdef double positive_prediction, negative_prediction
    cdef flt weight
    cdef flt *user_repr
    cdef flt *pos_it_repr
    cdef flt *neg_it_repr
    cdef unsigned int[::1] random_states
    cdef omp_lock_t reg_lock

    random_states = (random_state
                     .randint(0, np.iinfo(np.int32).max, size=num_threads)
                     .astype(np.uint32))

    no_examples = Y.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_bpr: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_bpr: OMP lock initialised\n")

    with nogil, parallel(num_threads=num_threads):
        user_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        pos_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        neg_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

        if user_repr == NULL or pos_it_repr == NULL or neg_it_repr == NULL:
            fprintf(stderr, b"[ERROR] fit_bpr: malloc failed for thread buffer\n")
        else:
            for i in prange(no_examples, schedule='dynamic'):
                row = shuffle_indices[i]

                if not Y[row] > 0:
                    continue

                weight           = sample_weight[row]
                user_id          = user_ids[row]
                positive_item_id = item_ids[row]

                # Sample a negative item (not in user's positives)
                for j in range(no_examples):
                    negative_item_id = item_ids[
                        rand_r(&random_states[threadid()]) % no_examples]
                    if not in_positives(negative_item_id, user_id, interactions):
                        break

                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_id, model.user_scale, user_repr)
                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, positive_item_id, model.item_scale,
                                       pos_it_repr)
                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, negative_item_id, model.item_scale,
                                       neg_it_repr)

                positive_prediction = compute_prediction_from_repr(
                    user_repr, pos_it_repr, model.no_components)
                negative_prediction = compute_prediction_from_repr(
                    user_repr, neg_it_repr, model.no_components)

                warp_update(
                    weight * (1.0 - sigmoid(<flt>(positive_prediction
                                                  - negative_prediction))),
                    item_features, user_features,
                    user_id, positive_item_id, negative_item_id,
                    user_repr, pos_it_repr, neg_it_repr,
                    model, item_alpha, user_alpha)

                if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
                    locked_regularize(model, item_alpha, user_alpha,
                                      &reg_lock, MAX_REG_SCALE)

        free(user_repr)
        free(pos_it_repr)
        free(neg_it_repr)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_bpr: epoch complete\n")


Writing ./cy/_cy_fit_bpr.pyx


In [10]:
%%writefile ./cy/_cy_fit_logistic.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_logistic.pyx
# Logistic-loss SGD training kernel for arycolbring.
# Parallelised with OpenMP prange; each thread owns its own malloc'd buffers.

import numpy as np
from libc.stdlib cimport malloc, free
from libc.stdio  cimport fprintf, stderr
from cython.parallel cimport prange, parallel

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport sigmoid
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, \
                                 omp_init_lock, omp_destroy_lock

cdef double MAX_REG_SCALE = 1000000.0


def fit_logistic(CSRMatrix item_features,
                 CSRMatrix user_features,
                 int[::1] user_ids,
                 int[::1] item_ids,
                 flt[::1] Y,
                 flt[::1] sample_weight,
                 int[::1] shuffle_indices,
                 FastAryColBring model,
                 double learning_rate,
                 double item_alpha,
                 double user_alpha,
                 int num_threads):
    """
    One epoch of logistic-loss collaborative filtering.

    Each worker thread allocates its own (user_repr, it_repr) buffers.
    A shared OMP lock serialises the periodic full-regularisation flush.
    """
    fprintf(stderr,
            b"[DEBUG] fit_logistic: no_examples=%d num_threads=%d\n",
            Y.shape[0], num_threads)

    cdef int i, row, user_id, item_id, no_examples
    cdef double prediction, loss
    cdef int y
    cdef flt y_row, weight
    cdef flt *user_repr
    cdef flt *it_repr
    cdef omp_lock_t reg_lock

    no_examples = Y.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_logistic: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_logistic: OMP lock initialised\n")

    with nogil, parallel(num_threads=num_threads):
        user_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        it_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

        if user_repr == NULL or it_repr == NULL:
            fprintf(stderr, b"[ERROR] fit_logistic: malloc failed for thread buffer\n")
        else:
            for i in prange(no_examples, schedule='dynamic'):
                row     = shuffle_indices[i]
                user_id = user_ids[row]
                item_id = item_ids[row]
                weight  = sample_weight[row]

                compute_representation(user_features,
                                       model.user_features,
                                       model.user_biases,
                                       model, user_id,
                                       model.user_scale, user_repr)

                compute_representation(item_features,
                                       model.item_features,
                                       model.item_biases,
                                       model, item_id,
                                       model.item_scale, it_repr)

                prediction = sigmoid(
                    compute_prediction_from_repr(user_repr, it_repr,
                                                 model.no_components))

                y_row = Y[row]
                y = 1 if y_row > 0 else 0

                loss = weight * (prediction - y)

                update(loss, item_features, user_features,
                       user_id, item_id, user_repr, it_repr,
                       model, item_alpha, user_alpha)

                if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
                    locked_regularize(model, item_alpha, user_alpha,
                                      &reg_lock, MAX_REG_SCALE)

        free(user_repr)
        free(it_repr)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_logistic: epoch complete\n")


Writing ./cy/_cy_fit_logistic.pyx


In [11]:
%%writefile ./cy/_cy_fit_warp.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_warp.pyx
# WARP-loss SGD training kernel for arycolbring.
# Parallelised with OpenMP prange; per-thread PRNG seeds avoid contention.

import numpy as np
from libc.stdlib  cimport malloc, free
from libc.stdio   cimport fprintf, stderr
from cython.parallel cimport prange, parallel, threadid

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport rand_r, in_positives
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport warp_update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, \
                                 omp_init_lock, omp_destroy_lock

cdef extern from "math.h" nogil:
    double log(double)
    double floor(double)

cdef double MAX_REG_SCALE = 1000000.0


def fit_warp(CSRMatrix item_features,
             CSRMatrix user_features,
             CSRMatrix interactions,
             int[::1] user_ids,
             int[::1] item_ids,
             flt[::1] Y,
             flt[::1] sample_weight,
             int[::1] shuffle_indices,
             FastAryColBring model,
             double learning_rate,
             double item_alpha,
             double user_alpha,
             int num_threads,
             random_state):
    """
    One epoch of WARP-loss collaborative filtering.

    For each positive interaction, samples negatives until one violates the
    margin condition, then applies a rank-aware gradient update.
    Per-thread PRNG seeds are drawn from `random_state` before the parallel
    region to avoid GIL contention inside prange.
    """
    fprintf(stderr,
            b"[DEBUG] fit_warp: no_examples=%d num_threads=%d\n",
            Y.shape[0], num_threads)

    cdef int i, no_examples, user_id, positive_item_id, negative_item_id
    cdef int sampled, row
    cdef double positive_prediction, negative_prediction, loss
    cdef double MAX_LOSS = 10.0
    cdef flt weight
    cdef flt *user_repr
    cdef flt *pos_it_repr
    cdef flt *neg_it_repr
    cdef unsigned int[::1] random_states
    cdef omp_lock_t reg_lock

    random_states = (random_state
                     .randint(0, np.iinfo(np.int32).max, size=num_threads)
                     .astype(np.uint32))

    no_examples = Y.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_warp: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_warp: OMP lock initialised\n")

    with nogil, parallel(num_threads=num_threads):
        user_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        pos_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        neg_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

        if user_repr == NULL or pos_it_repr == NULL or neg_it_repr == NULL:
            fprintf(stderr, b"[ERROR] fit_warp: malloc failed for thread buffer\n")
        else:
            for i in prange(no_examples, schedule='dynamic'):
                row              = shuffle_indices[i]
                user_id          = user_ids[row]
                positive_item_id = item_ids[row]

                if not Y[row] > 0:
                    continue

                weight = sample_weight[row]

                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_id, model.user_scale, user_repr)
                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, positive_item_id, model.item_scale,
                                       pos_it_repr)

                positive_prediction = compute_prediction_from_repr(
                    user_repr, pos_it_repr, model.no_components)

                sampled = 0
                while sampled < model.max_sampled:
                    sampled += 1
                    negative_item_id = (rand_r(&random_states[threadid()])
                                        % item_features.rows)

                    compute_representation(item_features,
                                           model.item_features, model.item_biases,
                                           model, negative_item_id, model.item_scale,
                                           neg_it_repr)

                    negative_prediction = compute_prediction_from_repr(
                        user_repr, neg_it_repr, model.no_components)

                    if negative_prediction > positive_prediction - 1:
                        if in_positives(negative_item_id, user_id, interactions):
                            continue

                        loss = weight * log(
                            max(1.0,
                                floor((item_features.rows - 1) / <double>sampled)))

                        if loss > MAX_LOSS:
                            loss = MAX_LOSS

                        warp_update(loss, item_features, user_features,
                                    user_id, positive_item_id, negative_item_id,
                                    user_repr, pos_it_repr, neg_it_repr,
                                    model, item_alpha, user_alpha)
                        break

                if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
                    locked_regularize(model, item_alpha, user_alpha,
                                      &reg_lock, MAX_REG_SCALE)

        free(user_repr)
        free(pos_it_repr)
        free(neg_it_repr)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_warp: epoch complete\n")


Writing ./cy/_cy_fit_warp.pyx


In [12]:
%%writefile ./cy/_cy_fit_warp_kos.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_warp_kos.pyx
# WARP k-OS (k-th Order Statistic) training kernel for arycolbring.
# Selects the k-th highest-ranked positive item per user before applying WARP.

import numpy as np
from libc.stdlib  cimport malloc, free
from libc.stdio   cimport fprintf, stderr
from cython.parallel cimport prange, parallel, threadid

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport (rand_r, sample_range, in_positives,
                                  int_min, Pair, reverse_pair_compare, qsort)
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport warp_update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, \
                                 omp_init_lock, omp_destroy_lock

cdef extern from "math.h" nogil:
    double log(double)
    double floor(double)

cdef double MAX_REG_SCALE = 1000000.0


def fit_warp_kos(CSRMatrix item_features,
                 CSRMatrix user_features,
                 CSRMatrix data,
                 int[::1] user_ids,
                 int[::1] shuffle_indices,
                 FastAryColBring model,
                 double learning_rate,
                 double item_alpha,
                 double user_alpha,
                 int k,
                 int n,
                 int num_threads,
                 random_state):
    """
    One epoch of WARP-kOS collaborative filtering.

    For each user, `n` positive items are sampled; the k-th highest-scored
    one is selected as the anchor. Then WARP negative sampling proceeds as
    normal from that anchor.
    """
    fprintf(stderr,
            b"[DEBUG] fit_warp_kos: no_examples=%d k=%d n=%d num_threads=%d\n",
            user_ids.shape[0], k, n, num_threads)

    cdef int i, j, no_examples, user_id, positive_item_id, negative_item_id
    cdef int sampled, row, sampled_positive_item_id
    cdef int user_pids_start, user_pids_stop, no_positives
    cdef double positive_prediction, negative_prediction, sampled_positive_prediction
    cdef double loss, MAX_LOSS = 10.0
    cdef flt *user_repr
    cdef flt *pos_it_repr
    cdef flt *neg_it_repr
    cdef Pair *pos_pairs
    cdef unsigned int[::1] random_states
    cdef omp_lock_t reg_lock

    random_states = (random_state
                     .randint(0, np.iinfo(np.int32).max, size=num_threads)
                     .astype(np.uint32))

    no_examples = user_ids.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_warp_kos: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_warp_kos: OMP lock initialised\n")

    with nogil, parallel(num_threads=num_threads):
        user_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        pos_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        neg_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        pos_pairs   = <Pair*>malloc(sizeof(Pair) * n)

        if (user_repr == NULL or pos_it_repr == NULL
                or neg_it_repr == NULL or pos_pairs == NULL):
            fprintf(stderr, b"[ERROR] fit_warp_kos: malloc failed\n")
        else:
            for i in prange(no_examples, schedule='dynamic'):
                row     = shuffle_indices[i]
                user_id = user_ids[row]

                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_id, model.user_scale, user_repr)

                user_pids_start = data.get_row_start(user_id)
                user_pids_stop  = data.get_row_end(user_id)

                if user_pids_stop == user_pids_start:
                    continue

                # Sample up to n positives and score them
                no_positives = int_min(n, user_pids_stop - user_pids_start)
                for j in range(no_positives):
                    sampled_positive_item_id = data.indices[
                        sample_range(user_pids_start, user_pids_stop,
                                     &random_states[threadid()])]

                    compute_representation(item_features,
                                           model.item_features, model.item_biases,
                                           model, sampled_positive_item_id,
                                           model.item_scale, pos_it_repr)

                    sampled_positive_prediction = compute_prediction_from_repr(
                        user_repr, pos_it_repr, model.no_components)

                    pos_pairs[j].idx = sampled_positive_item_id
                    pos_pairs[j].val = <flt>sampled_positive_prediction

                # Pick the k-th order statistic (descending sort, then take index k-1)
                qsort(pos_pairs, no_positives, sizeof(Pair), reverse_pair_compare)

                positive_item_id    = pos_pairs[int_min(k, no_positives) - 1].idx
                positive_prediction = pos_pairs[int_min(k, no_positives) - 1].val

                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, positive_item_id, model.item_scale,
                                       pos_it_repr)

                # WARP negative sampling phase
                sampled = 0
                while sampled < model.max_sampled:
                    sampled += 1
                    negative_item_id = (rand_r(&random_states[threadid()])
                                        % item_features.rows)

                    compute_representation(item_features,
                                           model.item_features, model.item_biases,
                                           model, negative_item_id, model.item_scale,
                                           neg_it_repr)

                    negative_prediction = compute_prediction_from_repr(
                        user_repr, neg_it_repr, model.no_components)

                    if negative_prediction > positive_prediction - 1:
                        if in_positives(negative_item_id, user_id, data):
                            continue

                        loss = log(floor(
                            (item_features.rows - 1) / <double>sampled))
                        if loss > MAX_LOSS:
                            loss = MAX_LOSS

                        warp_update(loss, item_features, user_features,
                                    user_id, positive_item_id, negative_item_id,
                                    user_repr, pos_it_repr, neg_it_repr,
                                    model, item_alpha, user_alpha)
                        break

                if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
                    locked_regularize(model, item_alpha, user_alpha,
                                      &reg_lock, MAX_REG_SCALE)

        free(user_repr)
        free(pos_it_repr)
        free(neg_it_repr)
        free(pos_pairs)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_warp_kos: epoch complete\n")


Writing ./cy/_cy_fit_warp_kos.pyx


In [13]:
%%writefile ./cy/_cy_math.pxd

# _cy_math.pxd
# Inline math utilities for arycolbring Cython modules.
# All functions defined here are inlined into every module that cimports them.

from _cy_types cimport CSRMatrix, flt

# ── C standard library declarations ─────────────────────────────────────────

cdef extern from "math.h" nogil:
    double sqrt(double)
    double exp(double)
    double log(double)
    double floor(double)

cdef extern from "stdlib.h" nogil:
    void qsort(void *base, int nmemb, int size,
               int(*compar)(const void *, const void *)) nogil
    void* bsearch(const void *key, const void *base, int nmemb, int size,
                  int(*compar)(const void *, const void *)) nogil

# ── Pair struct (used by WARP-kOS) ──────────────────────────────────────────

cdef struct Pair:
    int idx
    flt val

# ── PRNG ────────────────────────────────────────────────────────────────────

cdef inline unsigned int temper(unsigned int x) nogil:
    cdef unsigned int and_1 = 0x9D2C5680
    cdef unsigned int and_2 = 0xEFC60000
    x = x ^ (x >> 11)
    x = x ^ (x << 7  & and_1)
    x = x ^ (x << 15 & and_2)
    x = x ^ (x >> 18)
    return x


cdef inline int rand_r(unsigned int *seed) nogil:
    seed[0] = seed[0] * 1103515245 + 12345
    return temper(seed[0]) / 2


cdef inline int sample_range(int min_val, int max_val, unsigned int *seed) nogil:
    cdef int val_range = max_val - min_val
    return min_val + (rand_r(seed) % val_range)

# ── Integer helpers ──────────────────────────────────────────────────────────

cdef inline int int_min(int x, int y) nogil:
    if x < y:
        return x
    return y


cdef inline int int_max(int x, int y) nogil:
    if x < y:
        return y
    return x

# ── Comparators for qsort / bsearch ─────────────────────────────────────────

cdef inline int int_compare(const void *a, const void *b) nogil:
    cdef int va = (<int*>a)[0]
    cdef int vb = (<int*>b)[0]
    if va > vb:
        return 1
    elif va < vb:
        return -1
    return 0


cdef inline int flt_compare(const void *a, const void *b) nogil:
    cdef flt va = (<flt*>a)[0]
    cdef flt vb = (<flt*>b)[0]
    if va > vb:
        return 1
    elif va < vb:
        return -1
    return 0


cdef inline int reverse_pair_compare(const void *a, const void *b) nogil:
    cdef flt diff = (<Pair*>a).val - (<Pair*>b).val
    if diff < 0:
        return 1
    return -1

# ── Sigmoid activation ───────────────────────────────────────────────────────

cdef inline flt sigmoid(flt v) nogil:
    return <flt>(1.0 / (1.0 + exp(-v)))

# ── Positives lookup (binary search in sorted CSR row) ───────────────────────

cdef inline int in_positives(int item_id,
                              int user_id,
                              CSRMatrix interactions) nogil:
    """
    Return 1 if item_id is in the sorted indices of user_id's CSR row.
    Uses bsearch for O(log k) lookup where k = nnz per row.
    """
    cdef int start_idx = interactions.get_row_start(user_id)
    cdef int stop_idx  = interactions.get_row_end(user_id)

    if bsearch(&item_id,
               &interactions.indices[start_idx],
               stop_idx - start_idx,
               sizeof(int),
               int_compare) == NULL:
        return 0
    return 1


Writing ./cy/_cy_math.pxd


In [14]:
%%writefile ./cy/_cy_math.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_math.pyx
# All math utilities are declared as inline cdef in _cy_math.pxd.
# This stub exists so Cython builds _cy_math as an extension module;
# the pxd body is inlined into every module that cimports from it.

# No standalone symbols needed here — everything is inline in the pxd.


Writing ./cy/_cy_math.pyx


In [15]:
%%writefile ./cy/_cy_predict.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_predict.pyx
# Prediction kernels for arycolbring: pointwise scores and item ranks.
# Both are parallelised with OpenMP prange.

from libc.stdlib  cimport malloc, free
from libc.stdio   cimport fprintf, stderr
from cython.parallel cimport prange, parallel

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport in_positives, int_max
from _cy_representation cimport compute_representation, compute_prediction_from_repr


def predict_arycolbring(CSRMatrix item_features,
                        CSRMatrix user_features,
                        int[::1] user_ids,
                        int[::1] item_ids,
                        flt[::1] predictions,
                        FastAryColBring model,
                        int num_threads):
    """
    Compute pointwise prediction scores for (user_id, item_id) pairs.

    Parameters
    ----------
    item_features  : CSRMatrix  [n_item_features × n_item_feat_cols]
    user_features  : CSRMatrix  [n_user_features × n_user_feat_cols]
    user_ids       : int32 array, length = n_examples
    item_ids       : int32 array, length = n_examples
    predictions    : float32 array, length = n_examples  (output, written in-place)
    model          : FastAryColBring  holding all embedding state
    num_threads    : int  ≥ 1
    """
    fprintf(stderr,
            b"[DEBUG] predict_arycolbring: n_examples=%d num_threads=%d\n",
            predictions.shape[0], num_threads)

    cdef int i, no_examples
    cdef flt *user_repr
    cdef flt *it_repr

    no_examples = predictions.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] predict_arycolbring: no_examples=0, nothing to score\n")
        return

    with nogil, parallel(num_threads=num_threads):
        user_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        it_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

        if user_repr == NULL or it_repr == NULL:
            fprintf(stderr, b"[ERROR] predict_arycolbring: malloc failed\n")
        else:
            for i in prange(no_examples, schedule='static'):
                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_ids[i],
                                       model.user_scale, user_repr)
                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, item_ids[i],
                                       model.item_scale, it_repr)

                predictions[i] = compute_prediction_from_repr(
                    user_repr, it_repr, model.no_components)

        free(user_repr)
        free(it_repr)

    fprintf(stderr, b"[DEBUG] predict_arycolbring: scoring complete\n")


def predict_ranks(CSRMatrix item_features,
                  CSRMatrix user_features,
                  CSRMatrix test_interactions,
                  CSRMatrix train_interactions,
                  flt[::1]  ranks,
                  FastAryColBring model,
                  int num_threads):
    """
    Compute the rank of every test-positive item for each user.

    For each user the routine:
      1. Scores the user's test-positive items.
      2. Scores ALL catalogue items (excluding train positives).
      3. Counts how many catalogue items outscore each test positive.
         That count is the item's rank (lower = better recommendation).

    Parameters
    ----------
    item_features     : CSRMatrix
    user_features     : CSRMatrix
    test_interactions : CSRMatrix   — positives to rank
    train_interactions: CSRMatrix   — positives to skip during ranking
    ranks             : float32 array matching test_interactions.data  (output)
    model             : FastAryColBring
    num_threads       : int ≥ 1
    """
    fprintf(stderr,
            b"[DEBUG] predict_ranks: n_users=%d num_threads=%d\n",
            test_interactions.rows, num_threads)

    cdef int i, j, user_id, item_id, predictions_size
    cdef int row_start, row_stop
    cdef flt *user_repr
    cdef flt *it_repr
    cdef int  *test_item_ids_buf
    cdef flt  *test_preds_buf
    cdef flt   prediction

    # Determine maximum row width (for buffer sizing)
    predictions_size = 0
    for user_id in range(test_interactions.rows):
        predictions_size = int_max(
            predictions_size,
            test_interactions.get_row_end(user_id)
            - test_interactions.get_row_start(user_id))

    fprintf(stderr,
            b"[DEBUG] predict_ranks: max_row_width=%d\n", predictions_size)

    if predictions_size == 0:
        fprintf(stderr, b"[WARN] predict_ranks: no test interactions found\n")
        return

    with nogil, parallel(num_threads=num_threads):
        user_repr       = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        it_repr         = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        test_item_ids_buf = <int*>malloc(sizeof(int) * predictions_size)
        test_preds_buf  = <flt*>malloc(sizeof(flt) * predictions_size)

        if (user_repr == NULL or it_repr == NULL
                or test_item_ids_buf == NULL or test_preds_buf == NULL):
            fprintf(stderr, b"[ERROR] predict_ranks: malloc failed\n")
        else:
            for user_id in prange(test_interactions.rows, schedule='dynamic'):
                row_start = test_interactions.get_row_start(user_id)
                row_stop  = test_interactions.get_row_end(user_id)

                if row_stop == row_start:
                    continue  # no test interactions for this user

                # Step 1 – compute user representation
                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_id,
                                       model.user_scale, user_repr)

                # Step 2 – score every test-positive item for this user
                for i in range(row_stop - row_start):
                    item_id = test_interactions.indices[row_start + i]
                    compute_representation(item_features,
                                           model.item_features, model.item_biases,
                                           model, item_id, model.item_scale, it_repr)
                    test_item_ids_buf[i] = item_id
                    test_preds_buf[i]    = compute_prediction_from_repr(
                        user_repr, it_repr, model.no_components)

                # Step 3 – count catalogue items that beat each test positive
                for item_id in range(test_interactions.cols):
                    if in_positives(item_id, user_id, train_interactions):
                        continue  # skip known train positives

                    compute_representation(item_features,
                                           model.item_features, model.item_biases,
                                           model, item_id, model.item_scale, it_repr)
                    prediction = compute_prediction_from_repr(
                        user_repr, it_repr, model.no_components)

                    for i in range(row_stop - row_start):
                        if item_id != test_item_ids_buf[i] and prediction >= test_preds_buf[i]:
                            ranks[row_start + i] += 1.0

        free(user_repr)
        free(it_repr)
        free(test_item_ids_buf)
        free(test_preds_buf)

    fprintf(stderr, b"[DEBUG] predict_ranks: ranking complete\n")


Writing ./cy/_cy_predict.pyx


In [16]:
%%writefile ./cy/_cy_regularize.pxd

# _cy_regularize.pxd
# Inline L2 regularisation helpers for arycolbring.
# regularize()        – flush accumulated lazy scale factor across all weights.
# locked_regularize() – thread-safe version using an OMP lock pointer.

from _cy_types cimport FastAryColBring, flt

cdef extern from "omp.h" nogil:
    ctypedef struct omp_lock_t:
        pass
    void omp_init_lock(omp_lock_t *) nogil
    void omp_destroy_lock(omp_lock_t *) nogil
    void omp_set_lock(omp_lock_t *) nogil
    void omp_unset_lock(omp_lock_t *) nogil


cdef inline void regularize(FastAryColBring model,
                             double item_alpha,
                             double user_alpha) nogil:
    """
    Flush the accumulated lazy-regularisation scale factors so that
    item_scale and user_scale are reset to 1.0.
    Every weight is divided by its current scale value.
    """
    cdef int i, j
    cdef int no_item_features = model.item_features.shape[0]
    cdef int no_user_features = model.user_features.shape[0]

    for i in range(no_item_features):
        for j in range(model.no_components):
            model.item_features[i, j] /= model.item_scale
        model.item_biases[i] /= model.item_scale

    for i in range(no_user_features):
        for j in range(model.no_components):
            model.user_features[i, j] /= model.user_scale
        model.user_biases[i] /= model.user_scale

    model.item_scale = 1.0
    model.user_scale = 1.0


cdef inline void locked_regularize(FastAryColBring model,
                                   double item_alpha,
                                   double user_alpha,
                                   omp_lock_t *lock,
                                   double MAX_REG_SCALE) nogil:
    """
    Thread-safe regularisation flush.
    Acquires *lock*, re-checks the threshold (another thread may have flushed
    already) and flushes if needed, then releases *lock*.
    """
    omp_set_lock(lock)
    if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
        regularize(model, item_alpha, user_alpha)
    omp_unset_lock(lock)


Writing ./cy/_cy_regularize.pxd


In [17]:
%%writefile ./cy/_cy_regularize.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_regularize.pyx
# All regularise logic is inline in _cy_regularize.pxd.


Writing ./cy/_cy_regularize.pyx


In [18]:
%%writefile ./cy/_cy_representation.pxd

# _cy_representation.pxd
# Inline functions for computing latent representations and dot-product
# predictions for arycolbring collaborative filtering.

from _cy_types cimport CSRMatrix, FastAryColBring, flt


cdef inline void compute_representation(CSRMatrix features,
                                        flt[:, ::1] feature_embeddings,
                                        flt[::1]    feature_biases,
                                        FastAryColBring model,
                                        int  row_id,
                                        double scale,
                                        flt  *representation) nogil:
    """
    Accumulate the weighted embedding and bias for a given row (user or item).

    The output `representation` is a C array of length (no_components + 1):
      - indices 0 .. no_components-1 : latent factor values
      - index   no_components         : bias term
    """
    cdef int i, j, start_index, stop_index, feature
    cdef flt feature_weight

    start_index = features.get_row_start(row_id)
    stop_index  = features.get_row_end(row_id)

    # Zero-initialise output buffer
    for i in range(model.no_components + 1):
        representation[i] = 0.0

    for i in range(start_index, stop_index):
        feature        = features.indices[i]
        feature_weight = <flt>(features.data[i] * scale)

        for j in range(model.no_components):
            representation[j] += feature_weight * feature_embeddings[feature, j]

        # Bias sits at position no_components
        representation[model.no_components] += feature_weight * feature_biases[feature]


cdef inline flt compute_prediction_from_repr(flt *user_repr,
                                             flt *item_repr,
                                             int  no_components) nogil:
    """
    Compute the score for (user, item) as:
        user_bias + item_bias + dot(user_latent, item_latent)
    """
    cdef int i
    cdef flt result

    # Bias terms
    result = user_repr[no_components] + item_repr[no_components]

    # Dot product of latent factors
    for i in range(no_components):
        result += user_repr[i] * item_repr[i]

    return result


Writing ./cy/_cy_representation.pxd


In [19]:
%%writefile ./cy/_cy_representation.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_representation.pyx
# All logic is inline in _cy_representation.pxd.
# This stub exists to satisfy Cython's build system.


Writing ./cy/_cy_representation.pyx


In [20]:
%%writefile ./cy/_cy_types.pxd

# _cy_types.pxd
# Shared type declarations for arycolbring Cython modules.
# cimport this file in any .pyx that needs CSRMatrix or FastAryColBring.

ctypedef float flt


cdef class CSRMatrix:
    """
    Lightweight wrapper around a scipy CSR matrix for nogil access.
    Exposes indices/indptr/data as typed memory views.
    """
    cdef int[::1] indices
    cdef int[::1] indptr
    cdef flt[::1] data

    cdef int rows
    cdef int cols
    cdef int nnz

    cdef int get_row_start(self, int row) nogil
    cdef int get_row_end(self, int row) nogil


cdef class FastAryColBring:
    """
    Holds all model state (embeddings, gradients, momentum) for
    the arycolbring collaborative filtering model.
    All fields are typed memory views for direct C-level access.
    """
    cdef flt[:, ::1] item_features
    cdef flt[:, ::1] item_feature_gradients
    cdef flt[:, ::1] item_feature_momentum

    cdef flt[::1] item_biases
    cdef flt[::1] item_bias_gradients
    cdef flt[::1] item_bias_momentum

    cdef flt[:, ::1] user_features
    cdef flt[:, ::1] user_feature_gradients
    cdef flt[:, ::1] user_feature_momentum

    cdef flt[::1] user_biases
    cdef flt[::1] user_bias_gradients
    cdef flt[::1] user_bias_momentum

    cdef int no_components
    cdef int adadelta
    cdef flt learning_rate
    cdef flt rho
    cdef flt eps
    cdef int max_sampled

    cdef double item_scale
    cdef double user_scale


Writing ./cy/_cy_types.pxd


In [21]:
%%writefile ./cy/_cy_types.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_types.pyx
# Implementation of CSRMatrix and FastAryColBring cdef classes.

from libc.stdio cimport fprintf, stderr


cdef class CSRMatrix:
    """
    Thin wrapper over a scipy CSR sparse matrix.
    Provides nogil-safe row slicing via get_row_start / get_row_end.
    """

    def __init__(self, csr_matrix):
        fprintf(stderr, b"[DEBUG] CSRMatrix.__init__: wrapping sparse matrix\n")

        self.indices = csr_matrix.indices
        self.indptr  = csr_matrix.indptr
        self.data    = csr_matrix.data

        self.rows, self.cols = csr_matrix.shape
        self.nnz = len(self.data)

        fprintf(stderr,
                b"[DEBUG] CSRMatrix.__init__: rows=%d cols=%d nnz=%d\n",
                self.rows, self.cols, self.nnz)

    cdef int get_row_start(self, int row) nogil:
        return self.indptr[row]

    cdef int get_row_end(self, int row) nogil:
        return self.indptr[row + 1]


cdef class FastAryColBring:
    """
    Central model-state container.
    All embedding matrices and their optimiser accumulators live here.
    Passed by reference to every Cython kernel so they can mutate
    the arrays in-place without any Python overhead.
    """

    def __init__(self,
                 flt[:, ::1] item_features,
                 flt[:, ::1] item_feature_gradients,
                 flt[:, ::1] item_feature_momentum,
                 flt[::1]    item_biases,
                 flt[::1]    item_bias_gradients,
                 flt[::1]    item_bias_momentum,
                 flt[:, ::1] user_features,
                 flt[:, ::1] user_feature_gradients,
                 flt[:, ::1] user_feature_momentum,
                 flt[::1]    user_biases,
                 flt[::1]    user_bias_gradients,
                 flt[::1]    user_bias_momentum,
                 int         no_components,
                 int         adadelta,
                 flt         learning_rate,
                 flt         rho,
                 flt         epsilon,
                 int         max_sampled):

        fprintf(stderr,
                b"[DEBUG] FastAryColBring.__init__: no_components=%d adadelta=%d\n",
                no_components, adadelta)

        self.item_features           = item_features
        self.item_feature_gradients  = item_feature_gradients
        self.item_feature_momentum   = item_feature_momentum
        self.item_biases             = item_biases
        self.item_bias_gradients     = item_bias_gradients
        self.item_bias_momentum      = item_bias_momentum

        self.user_features           = user_features
        self.user_feature_gradients  = user_feature_gradients
        self.user_feature_momentum   = user_feature_momentum
        self.user_biases             = user_biases
        self.user_bias_gradients     = user_bias_gradients
        self.user_bias_momentum      = user_bias_momentum

        self.no_components  = no_components
        self.learning_rate  = learning_rate
        self.rho            = rho
        self.eps            = epsilon
        self.item_scale     = 1.0
        self.user_scale     = 1.0
        self.adadelta       = adadelta
        self.max_sampled    = max_sampled

        fprintf(stderr, b"[DEBUG] FastAryColBring.__init__: done\n")


Writing ./cy/_cy_types.pyx


In [22]:
%%writefile ./cy/_cy_update.pxd

# _cy_update.pxd
# Inline SGD update kernels for arycolbring.
# Supports both Adagrad and Adadelta learning-rate schedules.

from _cy_types cimport CSRMatrix, FastAryColBring, flt

cdef extern from "math.h" nogil:
    double sqrt(double)


# ── Bias update ──────────────────────────────────────────────────────────────

cdef inline double update_biases(CSRMatrix feature_indices,
                                 int start,
                                 int stop,
                                 flt[::1] biases,
                                 flt[::1] gradients,
                                 flt[::1] momentum,
                                 double gradient,
                                 int    adadelta,
                                 double learning_rate,
                                 double alpha,
                                 flt    rho,
                                 flt    eps) nogil:
    """
    Apply one SGD step on bias terms using Adagrad or Adadelta.
    Returns sum of per-feature local learning rates.
    """
    cdef int i, feature
    cdef double feature_weight, local_lr, update, sum_lr = 0.0

    if adadelta:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            gradients[feature]  = (rho * gradients[feature]
                                   + (1.0 - rho) * (feature_weight * gradient) ** 2)
            local_lr            = (sqrt(momentum[feature] + eps)
                                   / sqrt(gradients[feature] + eps))
            update              = local_lr * gradient * feature_weight
            momentum[feature]   = (rho * momentum[feature]
                                   + (1.0 - rho) * update ** 2)
            biases[feature]    -= <flt>update
            # Lazy L2 regularisation: inflate scale, not every weight
            biases[feature]    *= <flt>(1.0 + alpha * local_lr)
            sum_lr             += local_lr
    else:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            local_lr           = learning_rate / sqrt(gradients[feature])
            biases[feature]   -= <flt>(local_lr * feature_weight * gradient)
            gradients[feature] += <flt>((gradient * feature_weight) ** 2)
            biases[feature]   *= <flt>(1.0 + alpha * local_lr)
            sum_lr            += local_lr

    return sum_lr


# ── Embedding update ─────────────────────────────────────────────────────────

cdef inline double update_features(CSRMatrix  feature_indices,
                                   flt[:, ::1] features,
                                   flt[:, ::1] gradients,
                                   flt[:, ::1] momentum,
                                   int    component,
                                   int    start,
                                   int    stop,
                                   double gradient,
                                   int    adadelta,
                                   double learning_rate,
                                   double alpha,
                                   flt    rho,
                                   flt    eps) nogil:
    """
    Apply one SGD step for a single latent component across all active features.
    Returns sum of per-feature local learning rates.
    """
    cdef int i, feature
    cdef double feature_weight, local_lr, update, sum_lr = 0.0

    if adadelta:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            gradients[feature, component]  = (
                rho * gradients[feature, component]
                + (1.0 - rho) * (feature_weight * gradient) ** 2)
            local_lr = (sqrt(momentum[feature, component] + eps)
                        / sqrt(gradients[feature, component] + eps))
            update   = local_lr * gradient * feature_weight
            momentum[feature, component] = (
                rho * momentum[feature, component] + (1.0 - rho) * update ** 2)
            features[feature, component] -= <flt>update
            features[feature, component] *= <flt>(1.0 + alpha * local_lr)
            sum_lr += local_lr
    else:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            local_lr = learning_rate / sqrt(gradients[feature, component])
            features[feature, component]  -= <flt>(local_lr * feature_weight * gradient)
            gradients[feature, component] += <flt>((gradient * feature_weight) ** 2)
            features[feature, component]  *= <flt>(1.0 + alpha * local_lr)
            sum_lr += local_lr

    return sum_lr


# ── Logistic gradient step ────────────────────────────────────────────────────

cdef inline void update(double loss,
                        CSRMatrix item_features,
                        CSRMatrix user_features,
                        int user_id,
                        int item_id,
                        flt *user_repr,
                        flt *it_repr,
                        FastAryColBring model,
                        double item_alpha,
                        double user_alpha) nogil:
    """
    Apply the logistic gradient update for a single (user, item) pair.
    """
    cdef int i
    cdef int item_start = item_features.get_row_start(item_id)
    cdef int item_stop  = item_features.get_row_end(item_id)
    cdef int user_start = user_features.get_row_start(user_id)
    cdef int user_stop  = user_features.get_row_end(user_id)
    cdef double avg_lr  = 0.0
    cdef flt item_component, user_component

    avg_lr += update_biases(item_features, item_start, item_stop,
                            model.item_biases, model.item_bias_gradients,
                            model.item_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            item_alpha, model.rho, model.eps)

    avg_lr += update_biases(user_features, user_start, user_stop,
                            model.user_biases, model.user_bias_gradients,
                            model.user_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            user_alpha, model.rho, model.eps)

    for i in range(model.no_components):
        item_component = it_repr[i]
        user_component = user_repr[i]

        avg_lr += update_features(item_features, model.item_features,
                                  model.item_feature_gradients,
                                  model.item_feature_momentum,
                                  i, item_start, item_stop,
                                  loss * user_component,
                                  model.adadelta, model.learning_rate,
                                  item_alpha, model.rho, model.eps)

        avg_lr += update_features(user_features, model.user_features,
                                  model.user_feature_gradients,
                                  model.user_feature_momentum,
                                  i, user_start, user_stop,
                                  loss * item_component,
                                  model.adadelta, model.learning_rate,
                                  user_alpha, model.rho, model.eps)

    avg_lr /= ((model.no_components + 1) * (item_stop - item_start)
               + (model.no_components + 1) * (user_stop - user_start))

    model.item_scale *= 1.0 + item_alpha * avg_lr
    model.user_scale *= 1.0 + user_alpha * avg_lr


# ── WARP gradient step ────────────────────────────────────────────────────────

cdef inline void warp_update(double loss,
                             CSRMatrix item_features,
                             CSRMatrix user_features,
                             int user_id,
                             int positive_item_id,
                             int negative_item_id,
                             flt *user_repr,
                             flt *pos_it_repr,
                             flt *neg_it_repr,
                             FastAryColBring model,
                             double item_alpha,
                             double user_alpha) nogil:
    """
    Apply the WARP pairwise gradient update.
    Positive item receives a push-up; negative item receives a push-down.
    """
    cdef int i
    cdef int pos_start  = item_features.get_row_start(positive_item_id)
    cdef int pos_stop   = item_features.get_row_end(positive_item_id)
    cdef int neg_start  = item_features.get_row_start(negative_item_id)
    cdef int neg_stop   = item_features.get_row_end(negative_item_id)
    cdef int user_start = user_features.get_row_start(user_id)
    cdef int user_stop  = user_features.get_row_end(user_id)
    cdef double avg_lr  = 0.0
    cdef flt pos_comp, neg_comp, user_comp

    # Bias updates
    avg_lr += update_biases(item_features, pos_start, pos_stop,
                            model.item_biases, model.item_bias_gradients,
                            model.item_bias_momentum,
                            -loss, model.adadelta, model.learning_rate,
                            item_alpha, model.rho, model.eps)

    avg_lr += update_biases(item_features, neg_start, neg_stop,
                            model.item_biases, model.item_bias_gradients,
                            model.item_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            item_alpha, model.rho, model.eps)

    avg_lr += update_biases(user_features, user_start, user_stop,
                            model.user_biases, model.user_bias_gradients,
                            model.user_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            user_alpha, model.rho, model.eps)

    # Embedding updates
    for i in range(model.no_components):
        user_comp = user_repr[i]
        pos_comp  = pos_it_repr[i]
        neg_comp  = neg_it_repr[i]

        avg_lr += update_features(item_features, model.item_features,
                                  model.item_feature_gradients,
                                  model.item_feature_momentum,
                                  i, pos_start, pos_stop,
                                  -loss * user_comp,
                                  model.adadelta, model.learning_rate,
                                  item_alpha, model.rho, model.eps)

        avg_lr += update_features(item_features, model.item_features,
                                  model.item_feature_gradients,
                                  model.item_feature_momentum,
                                  i, neg_start, neg_stop,
                                  loss * user_comp,
                                  model.adadelta, model.learning_rate,
                                  item_alpha, model.rho, model.eps)

        avg_lr += update_features(user_features, model.user_features,
                                  model.user_feature_gradients,
                                  model.user_feature_momentum,
                                  i, user_start, user_stop,
                                  loss * (neg_comp - pos_comp),
                                  model.adadelta, model.learning_rate,
                                  user_alpha, model.rho, model.eps)

    avg_lr /= ((model.no_components + 1) * (user_stop - user_start)
               + (model.no_components + 1) * (pos_stop - pos_start)
               + (model.no_components + 1) * (neg_stop - neg_start))

    model.item_scale *= 1.0 + item_alpha * avg_lr
    model.user_scale *= 1.0 + user_alpha * avg_lr


Writing ./cy/_cy_update.pxd


In [23]:
%%writefile ./cy/_cy_update.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_update.pyx
# All update logic is inline in _cy_update.pxd.


Writing ./cy/_cy_update.pyx


In [24]:
%%writefile ./setup.py

# setup.py
"""
Build script for arycolbring Cython extensions.

Each Cython module is compiled as a separate shared library so they can
be debugged independently.  All modules are compiled with:
  - OpenMP for prange parallelism
  - aggressive optimisation flags
  - boundscheck / wraparound disabled at the compiler level

Usage
-----
    # Compile in-place (development)
    python setup.py build_ext --inplace

    # Install
    pip install .

Requirements
------------
  - Cython >= 0.29 or 3.x
  - A C compiler with OpenMP support:
      Linux  : GCC  (gcc -fopenmp)
      macOS  : clang via Homebrew  (brew install llvm; CC=clang-XX)
      Windows: MSVC (/openmp)
"""

import os
import sys
import numpy as np
from setuptools import setup, find_packages, Extension
from Cython.Build import cythonize

# ── compiler flags ────────────────────────────────────────────────────────────
if sys.platform == "win32":
    extra_compile_args = ["/O2", "/openmp"]
    extra_link_args    = []
    omp_lib            = []
elif sys.platform == "darwin":
    # Homebrew LLVM clang supports -fopenmp; Apple clang does not.
    extra_compile_args = ["-O3", "-fopenmp", "-march=native"]
    extra_link_args    = ["-fopenmp"]
    omp_lib            = ["omp"]
else:  # Linux
    extra_compile_args = [
        "-O3",
        "-fopenmp",
        "-march=native",
        "-ffast-math",
    ]
    extra_link_args = ["-fopenmp"]
    omp_lib         = []

include_dirs = [np.get_include()]
cy_dir       = os.path.join("arycolbring", "cy")


def cy_ext(name: str, sources=None) -> Extension:
    """
    Create an Extension for a Cython module inside ``arycolbring/cy/``.

    Parameters
    ----------
    name    : dotted module name, e.g. ``"arycolbring.cy._cy_types"``
    sources : list of .pyx paths; defaults to the single .pyx matching *name*
    """
    if sources is None:
        module_file = name.split(".")[-1] + ".pyx"
        sources     = [os.path.join(cy_dir, module_file)]
    return Extension(
        name,
        sources=sources,
        include_dirs=include_dirs,
        extra_compile_args=extra_compile_args,
        extra_link_args=extra_link_args,
        libraries=omp_lib,
        language="c",
    )


# ── extension modules ─────────────────────────────────────────────────────────
# Order matters: shared types first so the .pxd is on the include path
# when downstream modules are compiled.

extensions = [
    cy_ext("arycolbring.cy._cy_types"),
    cy_ext("arycolbring.cy._cy_math"),
    cy_ext("arycolbring.cy._cy_representation"),
    cy_ext("arycolbring.cy._cy_update"),
    cy_ext("arycolbring.cy._cy_regularize"),
    cy_ext("arycolbring.cy._cy_fit_logistic"),
    cy_ext("arycolbring.cy._cy_fit_warp"),
    cy_ext("arycolbring.cy._cy_fit_bpr"),
    cy_ext("arycolbring.cy._cy_fit_warp_kos"),
    cy_ext("arycolbring.cy._cy_predict"),
    cy_ext("arycolbring.cy._cy_evaluate"),
]

# ── Cython compiler directives ────────────────────────────────────────────────
compiler_directives = {
    "boundscheck":      False,
    "wraparound":       False,
    "cdivision":        True,
    "initializedcheck": False,
    "nonecheck":        False,
    "embedsignature":   True,    # Allows introspection of compiled functions
    "language_level":   "3",
}

# ── setup ─────────────────────────────────────────────────────────────────────
setup(
    name="arycolbring",
    version="0.1.0",
    description=(
        "Ultra-optimised user-to-item collaborative filtering "
        "with Cython + OpenMP kernels"
    ),
    author="aryanto",
    python_requires=">=3.8",
    packages=find_packages(exclude=["tests*"]),
    package_data={
        "arycolbring":    ["config.ini"],
        "arycolbring.cy": ["*.pxd", "*.pyx"],
    },
    ext_modules=cythonize(
        extensions,
        compiler_directives=compiler_directives,
        annotate=True,      # produces _cy_*.html annotation files for profiling
        nthreads=4,         # parallel Cython transpilation (not OpenMP)
    ),
    include_dirs=include_dirs,
    install_requires=[
        "numpy>=1.21",
        "scipy>=1.7",
        "pandas>=1.3",
        "duckdb>=0.8",
        "tqdm>=4.60",
        "joblib>=1.1",
        "cython>=0.29",
        "seaborn>=0.12",
    ],
    zip_safe=False,
)


Writing ./setup.py


In [25]:
!python setup.py build_ext --inplace

Traceback (most recent call last):
  File "/kaggle/working/setup.py", line 126, in <module>
    ext_modules=cythonize(
                ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/Cython/Build/Dependencies.py", line 1010, in cythonize
    module_list, module_metadata = create_extension_list(
                                   ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/Cython/Build/Dependencies.py", line 845, in create_extension_list
    for file in nonempty(sorted(extended_iglob(filepattern)), "'%s' doesn't match any files" % filepattern):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/Cython/Build/Dependencies.py", line 117, in nonempty
    raise ValueError(error_msg)
ValueError: 'arycolbring/cy/_cy_types.pyx' doesn't match any files
